# Interactive comparison of FINN fire polygons

A standalone notebook for side-by-side comparison of the original FINN2 output (`.shp`) with the new `finn_py` output (`.gpkg`).  Uses **folium** for a zoom/pan interactive map with toggleable layers, click-through attribute popups, and disagreement highlighting.

This notebook is purely a viewer — it does not run the pipeline, doesn't import `finn_py`, and works on any two output files in the FINN-style schema.

Required columns on both inputs (the standard FINN2 schema):

```
polyid, fireid, cen_lon, cen_lat, acq_date_utc, area_sqkm,
alg_agg, v_lct, f_lct, v_tree, v_herb, v_bare, v_regnum, v_frp
```

Shapefile column truncation (`acq_date_utc` → `acq_date_u`, etc.) is handled automatically.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
import datetime as dt

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium import FeatureGroup, GeoJson, GeoJsonTooltip, GeoJsonPopup, LayerControl
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore', message='Geometry is in a geographic CRS')
print(f'geopandas {gpd.__version__},  folium {folium.__version__}')

### Edit your paths here
Two paths needed to Reference and New FINN processed fires SHP, GPKG or CSV.  CSV is accepted for either (point geometries will be built from `cen_lon`/`cen_lat`).

In [ ]:
# ============================================================
# EDIT THESE PATHS
# ============================================================
DATE = 2026270
# original FINNv2.5
#REF_PATH = Path(f'/glade/campaign/acom/acom-weather/emmons/emissions_finn/finn2.5.1nrt/processed_fires/out_modvrs_nrt_{DATE}_modlct_2023_modvcf_2023_regnum_frp.shp')
REF_PATH = Path(f'/glade/derecho/scratch/emmons/finn2.9nrt_output/out_v2.9nrt_{DATE}_modlct_2024_modvcf_2024_regnum.gpkg')
NEW_PATH = Path(f'/glade/derecho/scratch/emmons/finn2.9_mod3vrs_output/out_v2.9NRTmod3vrs_{DATE}_modlct_2024_modvcf_2024_regnum.gpkg')

# Optional: list of FIRMS AF input files (the same ones the pipeline ran on).
# Leave empty / set to [] to skip AF overlays.  Accepts .shp / .gpkg / .csv / .txt.
AF_PATHS = [
    Path(f'/glade/derecho/scratch/emmons/finnpy_inputs/ActiveFires/MODIS_C6_1_Global_MCD14DL_NRT_{DATE}.txt'),
    Path(f'/glade/derecho/scratch/emmons/finnpy_inputs/ActiveFires/SUOMI_VIIRS_C2_Global_VNP14IMGTDL_NRT_{DATE}.txt'),
    Path(f'/glade/derecho/scratch/emmons/finnpy_inputs/ActiveFires/J1_VIIRS_C2_Global_VJ114IMGTDL_NRT_{DATE}.txt'),
    Path(f'/glade/derecho/scratch/emmons/finnpy_inputs/ActiveFires/J2_VIIRS_C2_Global_VJ214IMGTDL_NRT_{DATE}.txt'),
]

# Where to save the HTML maps produced by this notebook (optional).
HTML_OUT_DIR = Path('./compare_maps_v2.9_3vrs').resolve()
HTML_OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for label, p in [('REF', REF_PATH), ('NEW', NEW_PATH)]:
    if not p.exists():
        raise FileNotFoundError(f'{label}_PATH not found: {p}')
    print(f'{label}: {p}  ({p.stat().st_size/1e6:.1f} MB)')

# AF paths are optional — warn but don't raise if any are missing.
for p in AF_PATHS:
    if Path(p).exists():
        print(f'AF : {p}  ({Path(p).stat().st_size/1e6:.1f} MB)')
    else:
        print(f'AF : {p}  ⚠ missing — will be skipped')
if not AF_PATHS:
    print('AF : (no AF_PATHS set — maps will show polygons only)')

## 2. Load both datasets

Handles `.shp`, `.gpkg`, or `.csv` on either side.  Shapefiles truncate column names to 10 chars, so I normalize `acq_date_u` → `acq_date_utc` to keep downstream code uniform.

In [ ]:
# Map any shapefile-truncated column names back to their canonical form.
SHP_COL_FIXUP = {
    'acq_date_u': 'acq_date_utc',
    'acq_date_l': 'acq_date_lst',
}

# Columns we want to compare numerically.  Loading from various GIS
# formats can leave these in surprising states — object dtype with
# literal '<NA>' strings is the worst-case after some shapefile or
# GPKG round-trips.  coerce_numeric normalises them all the same way.
INT_COLS   = ['polyid', 'fireid', 'alg_agg', 'v_lct', 'v_regnum']
FLOAT_COLS = ['cen_lon', 'cen_lat', 'area_sqkm', 'f_lct',
              'v_tree', 'v_herb', 'v_bare', 'v_frp']

def coerce_numeric(s, target='float'):
    """Coerce to nullable Int64 or float64, handling '<NA>' strings."""
    if s.dtype == 'object':
        s = s.replace({
            '<NA>': pd.NA, 'NA': pd.NA, 'nan': pd.NA, 'NaN': pd.NA,
            'None': pd.NA, '': pd.NA,
        })
    n = pd.to_numeric(s, errors='coerce')
    return n.astype('Int64') if target == 'int' else n

def load_finn_output(path: Path) -> gpd.GeoDataFrame:
    """Read .shp / .gpkg / .csv and normalise column names + types."""
    p = Path(path)
    if p.suffix.lower() == '.csv':
        df = pd.read_csv(p)
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df.cen_lon, df.cen_lat),
            crs='EPSG:4326',
        )
    else:
        gdf = gpd.read_file(p)
        if gdf.crs is None:
            gdf = gdf.set_crs('EPSG:4326')
        elif gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs('EPSG:4326')

    # Rename shapefile-truncated columns
    gdf = gdf.rename(columns=SHP_COL_FIXUP)

    # Coerce numeric columns to a consistent dtype, fixing '<NA>' etc.
    for c in INT_COLS:
        if c in gdf.columns:
            gdf[c] = coerce_numeric(gdf[c], 'int')
    for c in FLOAT_COLS:
        if c in gdf.columns:
            gdf[c] = coerce_numeric(gdf[c], 'float')

    # Pick whichever date column exists
    date_col = next(
        (c for c in ('acq_date_utc', 'acq_date_lst', 'acq_date_use')
         if c in gdf.columns),
        None,
    )
    if date_col is None:
        raise RuntimeError(f'{p}: no acq_date_* column found; '
                            f'have {sorted(gdf.columns)}')
    gdf[date_col] = pd.to_datetime(gdf[date_col]).dt.date
    gdf.attrs['date_col'] = date_col
    return gdf

ref = load_finn_output(REF_PATH)
new = load_finn_output(NEW_PATH)
print(f'REF: {len(ref):>7,} polygons, geom={ref.geometry.geom_type.iloc[0]:<10s} '
      f'date_col={ref.attrs["date_col"]}')
print(f'NEW: {len(new):>7,} polygons, geom={new.geometry.geom_type.iloc[0]:<10s} '
      f'date_col={new.attrs["date_col"]}')

In [ ]:
# Sanity check: do they have the columns we need to plot/compare?
REQUIRED = ['polyid', 'fireid', 'area_sqkm', 'alg_agg',
            'v_lct', 'v_tree', 'v_herb', 'v_bare', 'v_regnum', 'v_frp']
for label, gdf in [('REF', ref), ('NEW', new)]:
    missing = [c for c in REQUIRED if c not in gdf.columns]
    if missing:
        print(f'⚠️  {label} is missing columns: {missing}')
    else:
        print(f'✓ {label} has all required columns')

### Tag each polygon with a latitude band

FINN duplicates MODIS fires from the previous day in the tropics (MODIS swath gaps); 
Split everything by **Northern (>23.5°N)**, **Tropical (23.5°S–23.5°N)**, and **Southern (<23.5°S)** to check.

In [ ]:
BANDS = ['Northern (>23.5°N)', 'Tropical (23.5°S–23.5°N)', 'Southern (<23.5°S)']

def lat_band_of(lat: float) -> str:
    if pd.isna(lat):     return 'unknown'
    if lat >  23.5:      return BANDS[0]
    if lat < -23.5:      return BANDS[2]
    return BANDS[1]

for gdf in (ref, new):
    gdf['lat_band'] = gdf['cen_lat'].astype(float).map(lat_band_of)
    gdf['lat_band'] = pd.Categorical(gdf['lat_band'], categories=BANDS + ['unknown'])

print('Polygons per latitude band:')
side = pd.concat({
    'REF': ref['lat_band'].value_counts().reindex(BANDS, fill_value=0),
    'NEW': new['lat_band'].value_counts().reindex(BANDS, fill_value=0),
}, axis=1)
side['diff']     = side['NEW'] - side['REF']
side['diff_pct'] = (100 * side['diff'] / side['REF'].replace(0, np.nan)).round(1)
side

### Load AF detection inputs (optional)

If `AF_PATHS` was set above, load the underlying fire detections too.  Plotting them alongside the polygons makes it much easier to spot *why* polygons disagree — same detections clustered differently, different confidence filtering, LST vs UTC date boundaries, etc.

In [ ]:
def load_af_points(paths):
    """Load FIRMS AF detections from .shp / .gpkg / .csv / .txt files."""
    if not paths:
        return None
    frames = []
    for p in paths:
        p = Path(p)
        if not p.exists():
            continue
        if p.suffix.lower() in ('.shp', '.gpkg'):
            df = gpd.read_file(p)
        else:
            # FIRMS NRT .txt files are CSVs
            df = pd.read_csv(p)
        df.columns = [c.lower() if c != getattr(df, 'geometry', pd.Series()).name
                      else c for c in df.columns]
        if not isinstance(df, gpd.GeoDataFrame):
            if 'latitude' in df and 'longitude' in df:
                df = gpd.GeoDataFrame(
                    df,
                    geometry=gpd.points_from_xy(df.longitude, df.latitude),
                    crs='EPSG:4326',
                )
            else:
                print(f'  ⚠ {p}: no latitude/longitude columns — skipping')
                continue
        if df.crs is None:
            df = df.set_crs('EPSG:4326')
        elif df.crs.to_epsg() != 4326:
            df = df.to_crs('EPSG:4326')
        if 'acq_date' in df.columns:
            df['acq_date'] = pd.to_datetime(df['acq_date']).dt.date
        frames.append(df)
    if not frames:
        return None
    # Reindex onto column union so concat doesn't complain about ragged schemas
    # (MODIS has 'brightness'/'bright_t31', VIIRS has 'bright_ti4'/'bright_ti5').
    all_cols = set().union(*(set(f.columns) for f in frames))
    frames = [f.reindex(columns=list(all_cols)) for f in frames]
    return gpd.GeoDataFrame(
        pd.concat(frames, ignore_index=True), crs='EPSG:4326',
    )

af = load_af_points(AF_PATHS)
if af is not None:
    print(f'loaded {len(af):,} AF detections')
    if 'satellite' in af.columns:
        print(f'  by satellite: {af.satellite.value_counts().to_dict()}')
    if 'acq_date' in af.columns:
        print(f'  dates: {af.acq_date.min()} .. {af.acq_date.max()}')
else:
    print('no AF data loaded — maps will show polygons only')

## 3. Summary stats

In [ ]:
def summary(label, gdf):
    dc = gdf.attrs['date_col']
    d  = gdf[dc]
    return {
        'label':         label,
        'polygons':      len(gdf),
        'fires':         gdf['fireid'].nunique(),
        'total_area':    gdf['area_sqkm'].sum(),
        'date_min':      d.min(),
        'date_max':      d.max(),
        'unique_dates':  d.nunique(),
        'alg_agg=1':     int((gdf['alg_agg'] == 1).sum()),
        'alg_agg=2':     int((gdf['alg_agg'] == 2).sum()),
        'alg_agg NaN':   int(gdf['alg_agg'].isna().sum()),
        'mean v_tree':   gdf['v_tree'].mean(),
    }

pd.DataFrame([summary('REF', ref), summary('NEW', new)]).set_index('label').T

### By latitude band

Total polygon count and burned area per band, REF vs NEW.

In [ ]:
def band_table(metric_label, ref_fn, new_fn, fmt=lambda x: f'{x:,}'):
    rows = []
    for b in BANDS:
        r_val = ref_fn(ref[ref['lat_band'] == b])
        n_val = new_fn(new[new['lat_band'] == b])
        rows.append({
            'band': b,
            'REF':  r_val,
            'NEW':  n_val,
            'diff': n_val - r_val,
            'diff_pct': (100*(n_val - r_val) / r_val) if r_val else float('nan'),
        })
    # Append a total row
    rows.append({
        'band': 'TOTAL',
        'REF':  ref_fn(ref),
        'NEW':  new_fn(new),
        'diff': new_fn(new) - ref_fn(ref),
        'diff_pct': (100*(new_fn(new) - ref_fn(ref)) / ref_fn(ref))
                     if ref_fn(ref) else float('nan'),
    })
    t = pd.DataFrame(rows).set_index('band')
    t['diff_pct'] = t['diff_pct'].round(1)
    return t

print('Polygon counts:')
print(band_table('polygons', len, len))
print()
print('Total burned area (km²):')
area_tbl = band_table('area_km2',
                       lambda d: float(d['area_sqkm'].sum()),
                       lambda d: float(d['area_sqkm'].sum()))
for c in ['REF','NEW','diff']:
    area_tbl[c] = area_tbl[c].round(0).astype(int)
print(area_tbl)

## 4. Pick a viewing subset

Folium gets sluggish past ~10k polygons.  Default to **a single day** for a manageable initial view; bump up or filter spatially from there.  Both sides are subset the same way so the maps overlay apples-to-apples.

In [ ]:
# Find the shared date range and offer some defaults.
ref_dates = set(ref[ref.attrs['date_col']].unique())
new_dates = set(new[new.attrs['date_col']].unique())
shared = sorted(ref_dates & new_dates)
if not shared:
    raise RuntimeError('No shared dates between REF and NEW.')
print(f'{len(shared)} shared date(s): {shared[0]} .. {shared[-1]}')

# Pick the day with the most NEW polygons by default — busy days are
# the interesting ones for visual inspection.
polys_per_day = new[new.attrs['date_col']].value_counts()
busy_shared = [d for d in polys_per_day.index if d in shared]
default_day = busy_shared[0] if busy_shared else shared[0]
print(f'busiest shared day in NEW: {default_day} ({polys_per_day[default_day]:,} polygons)')

In [ ]:
# ============================================================
# EDIT: pick the subset to compare
# ============================================================
VIEW_DATE = default_day            # or e.g. dt.date(2020, 8, 15)
MAX_POLYGONS = 70000                # safety cap per side
#BBOX = (90, 0, 140, 40)                        # or (west, south, east, north) in lon/lat
BBOX = None                        # or (west, south, east, north) in lon/lat

def subset(gdf, date, bbox=None, max_n=None):
    dc = gdf.attrs['date_col']
    sub = gdf[gdf[dc] == date].copy()
    if bbox is not None:
        w, s, e, n = bbox
        sub = sub.cx[w:e, s:n]
    if max_n is not None and len(sub) > max_n:
        sub = sub.nlargest(max_n, 'area_sqkm')
    sub.attrs['date_col'] = dc
    return sub

ref_sub = subset(ref, VIEW_DATE, BBOX, MAX_POLYGONS)
new_sub = subset(new, VIEW_DATE, BBOX, MAX_POLYGONS)
print(f'subset for {VIEW_DATE}: REF={len(ref_sub):,}  NEW={len(new_sub):,}')

# Also subset AF to the same date (+ bbox if any).  AF date is always
# UTC; if the polygons are date_definition='LST' the AF date might be
# off by one at the dateline — widen by ±1 day if you see no overlap.
af_sub = None
if af is not None and 'acq_date' in af.columns:
    af_sub = af[af['acq_date'] == VIEW_DATE].copy()
    if BBOX is not None:
        w, s, e, n = BBOX
        af_sub = af_sub.cx[w:e, s:n]
    print(f'  AF detections on {VIEW_DATE}: {len(af_sub):,}')

## 5. Interactive overlay map

Both datasets on one map, toggleable from the layer-control widget in the top-right corner.  Click any polygon for a popup with the full attribute set; hover for a tooltip with the key fields.

Colours:

- **Blue polygons** = FINN2 reference
- **Red polygons**  = `finn_py` new
- **Yellow dots**   = AF detections (if `AF_PATHS` supplied)

Where the two polygon sets overlap you'll see purple-ish stripes because the fill is semi-transparent.  Any blue without red, or red without blue, is an interesting polygon to investigate — showing the yellow dots makes it obvious *why* the polygons differ (same dots clustered differently, or extra dots on one side from a different confidence filter, etc.).

In [ ]:
TOOLTIP_FIELDS = ['polyid','fireid','v_lct','v_tree','alg_agg','area_sqkm']
TOOLTIP_ALIASES= ['polyid','fireid','LCT class','tree %','alg_agg','area km²']

def gdf_to_layer(gdf, name, color, popup_fields=None,
                 fill_opacity=0.35, weight=0.5):
    """Wrap a GeoDataFrame as a folium FeatureGroup with tooltip+popup."""
    fg = FeatureGroup(name=name, show=True)
    # Build a lightweight GeoJSON — keep only fields we'll display.
    keep = list(set(TOOLTIP_FIELDS) | set(popup_fields or []))
    keep = [c for c in keep if c in gdf.columns]
    light = gdf[keep + [gdf.geometry.name]].copy()
    # Defensive type coercion so __geo_interface__ + json.dumps don't choke:
    #   - nullable Int / NaN floats -> object dtype with None for missing
    #   - datetime64 / date objects -> ISO-string
    for c in keep:
        if light[c].dtype.kind in 'iuf':
            light[c] = light[c].astype(object).where(light[c].notna(), None)
        elif light[c].dtype.kind == 'M':  # datetime64[ns]
            light[c] = light[c].dt.strftime('%Y-%m-%d')
        elif light[c].dtype == 'object':
            light[c] = light[c].map(
                lambda v: v.isoformat() if hasattr(v, 'isoformat') else v
            )
    GeoJson(
        light.__geo_interface__,
        style_function=lambda x, c=color, fo=fill_opacity, w=weight: {
            'color': c, 'fillColor': c,
            'fillOpacity': fo, 'weight': w,
        },
        highlight_function=lambda x: {'weight': 2, 'color': 'yellow'},
        tooltip=GeoJsonTooltip(
            fields=[f for f in TOOLTIP_FIELDS if f in keep],
            aliases=[a for f, a in zip(TOOLTIP_FIELDS, TOOLTIP_ALIASES) if f in keep],
            sticky=False,
        ),
        popup=GeoJsonPopup(
            fields=[f for f in (popup_fields or TOOLTIP_FIELDS) if f in keep],
        ) if popup_fields else None,
    ).add_to(fg)
    return fg

POPUP_FIELDS = ['polyid','fireid','area_sqkm','alg_agg',
                'v_lct','f_lct','v_tree','v_herb','v_bare','v_regnum','v_frp']

AF_TOOLTIP = [('acq_date','date'), ('acq_time','time'),
              ('satellite','sat'), ('confidence','conf'),
              ('frp','FRP'), ('daynight','D/N')]

def af_to_layer(af_gdf, name=None, color='black',
                fill_color='yellow', radius=3):
    """Wrap AF detection points as a folium FeatureGroup of CircleMarkers."""
    if af_gdf is None or not len(af_gdf):
        return None
    name = name or f'AF detections ({len(af_gdf):,})'
    fg = FeatureGroup(name=name, show=True)
    fields  = [c for c, _ in AF_TOOLTIP if c in af_gdf.columns]
    aliases = [a for c, a in AF_TOOLTIP if c in af_gdf.columns]
    light = af_gdf[fields + [af_gdf.geometry.name]].copy()
    for c in fields:
        if light[c].dtype.kind == 'M':
            light[c] = light[c].dt.strftime('%Y-%m-%d')
        elif light[c].dtype == 'object':
            light[c] = light[c].map(
                lambda v: v.isoformat() if hasattr(v, 'isoformat') else v
            )
        elif light[c].dtype.kind in 'iuf':
            light[c] = light[c].astype(object).where(light[c].notna(), None)
    GeoJson(
        light.__geo_interface__,
        marker=folium.CircleMarker(
            radius=radius, color=color, weight=1,
            fill=True, fillColor=fill_color, fillOpacity=0.7,
        ),
        tooltip=GeoJsonTooltip(fields=fields, aliases=aliases),
    ).add_to(fg)
    return fg

# Compute the map center + bounds from the union of both subsets.
all_bounds = pd.concat([ref_sub, new_sub]).total_bounds
center = [(all_bounds[1]+all_bounds[3])/2, (all_bounds[0]+all_bounds[2])/2]

m = folium.Map(location=center, zoom_start=6, tiles='OpenStreetMap')
gdf_to_layer(ref_sub, f'FINN Ref ({len(ref_sub):,})',
             'blue', POPUP_FIELDS).add_to(m)
gdf_to_layer(new_sub, f'FINN New ({len(new_sub):,})',
             'red',  POPUP_FIELDS).add_to(m)
af_layer = af_to_layer(af_sub)
if af_layer is not None:
    af_layer.add_to(m)
LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[all_bounds[1], all_bounds[0]], [all_bounds[3], all_bounds[2]]])

# Save a copy so it can be re-opened outside the notebook.
out_html = HTML_OUT_DIR / f'overlay_{VIEW_DATE}.html'
m.save(str(out_html))
print(f'saved {out_html}')
m

## 6. Highlight disagreements

Match each NEW polygon to its nearest REF polygon (by centroid, within the viewing subset), then highlight polygons that disagree on `alg_agg` or `v_lct`.  Yellow halos in the map below mark polygons worth investigating.

In [ ]:
# Per-day nearest-neighbor match (centroid-based).
def centroid(gdf):
    return gdf.geometry.centroid if gdf.geometry.geom_type.iloc[0] != 'Point' else gdf.geometry

ref_pts = np.column_stack([centroid(ref_sub).x, centroid(ref_sub).y])
new_pts = np.column_stack([centroid(new_sub).x, centroid(new_sub).y])
if len(ref_pts) == 0 or len(new_pts) == 0:
    raise RuntimeError('one side is empty — pick a different date or wider bbox')

tree = cKDTree(ref_pts)
dist_deg, idx = tree.query(new_pts, k=1)
# Crude km conversion (lat-corrected).
dist_km = dist_deg * 111.0 * np.cos(np.radians(new_pts[:, 1]))

match = pd.DataFrame({
    'new_polyid':  new_sub['polyid'].values,
    'ref_polyid':  ref_sub['polyid'].values[idx],
    'match_km':    dist_km,
    'cen_lat':     new_sub['cen_lat'].astype(float).values,
    'lat_band':    new_sub['lat_band'].astype(str).values,
})
for col in ['alg_agg','v_lct','v_tree','area_sqkm','v_frp']:
    if col in ref_sub.columns:
        match[f'ref_{col}'] = ref_sub[col].values[idx]
    if col in new_sub.columns:
        match[f'new_{col}'] = new_sub[col].values

# Both load_finn_output and the gpkg round-trip *should* have left
# alg_agg / v_lct as nullable Int64 already, but be defensive in case
# something downstream re-introduced strings or object dtype.
for c in ('ref_alg_agg', 'new_alg_agg', 'ref_v_lct', 'new_v_lct'):
    if c in match.columns:
        match[c] = coerce_numeric(match[c], 'int')

# Define disagreement: alg_agg flip OR v_lct mismatch (skip NaN pairs).
def both_valid(a, b):
    return a.notna() & b.notna()

alg_flip = both_valid(match.ref_alg_agg, match.new_alg_agg) & (
    match.ref_alg_agg != match.new_alg_agg)
lct_flip = both_valid(match.ref_v_lct, match.new_v_lct) & (
    match.ref_v_lct != match.new_v_lct)
match['disagree'] = alg_flip | lct_flip

print(f'OVERALL: {int(alg_flip.sum())} alg_agg flips, '
      f'{int(lct_flip.sum())} v_lct flips out of {len(match):,} matches')
print(f'  match distance: median={match.match_km.median():.3f} km, '
      f'max={match.match_km.max():.3f} km')

### Disagreements by latitude band

If the Tropical row has dramatically more disagreements than the other two bands, that's the duplication signal — extra REF polygons in the tropics get matched to whichever NEW polygon happened to be nearest, often producing spurious flips.  Look at the **median match distance** per band too: a much larger Tropical median means the nearest-neighbor matches are reaching across real fire boundaries to find a counterpart.

In [ ]:
rows = []
for b in BANDS + ['unknown']:
    sel = match['lat_band'] == b
    if not sel.any():
        continue
    sub = match[sel]
    alg_b = both_valid(sub.ref_alg_agg, sub.new_alg_agg) & (
        sub.ref_alg_agg != sub.new_alg_agg)
    lct_b = both_valid(sub.ref_v_lct, sub.new_v_lct) & (
        sub.ref_v_lct != sub.new_v_lct)
    rows.append({
        'band':        b,
        'matches':     len(sub),
        'median_km':   round(float(sub['match_km'].median()), 3),
        'max_km':      round(float(sub['match_km'].max()), 2),
        'alg_flips':   int(alg_b.sum()),
        'alg_flip_%':  round(100*alg_b.sum()/max(1,len(sub)), 2),
        'lct_flips':   int(lct_b.sum()),
        'lct_flip_%':  round(100*lct_b.sum()/max(1,len(sub)), 2),
    })
by_band = pd.DataFrame(rows).set_index('band')
by_band

In [ ]:
# Highlight map: gray base = agreeing polygons, red+thick outline
# = disagreements.  Both layers go through gdf_to_layer which
# filters columns down to displayable ones — anything else would
# choke json.dumps on e.g. datetime.date columns.
agree_ids   = match.loc[~match['disagree'], 'new_polyid'].tolist()
disagree_ids= match.loc[ match['disagree'], 'new_polyid'].tolist()
agree_sub    = new_sub[new_sub['polyid'].isin(agree_ids)]
disagree_sub = new_sub[new_sub['polyid'].isin(disagree_ids)]

m2 = folium.Map(location=center, zoom_start=6, tiles='OpenStreetMap')
gdf_to_layer(agree_sub, f'agree ({len(agree_sub):,})',
             'gray',  POPUP_FIELDS,
             fill_opacity=0.25, weight=0.3).add_to(m2)
if len(disagree_sub):
    gdf_to_layer(disagree_sub, f'disagree ({len(disagree_sub):,})',
                 'red', POPUP_FIELDS,
                 fill_opacity=0.50, weight=2).add_to(m2)
af_layer = af_to_layer(af_sub)
if af_layer is not None:
    af_layer.add_to(m2)
LayerControl(collapsed=False).add_to(m2)
m2.fit_bounds([[all_bounds[1], all_bounds[0]], [all_bounds[3], all_bounds[2]]])

out_html2 = HTML_OUT_DIR / f'disagree_{VIEW_DATE}.html'
m2.save(str(out_html2))
print(f'saved {out_html2}')
m2

## 7. Inspect specific polygons

Side-by-side attribute table for the largest disagreements.  Sort by `match_km` desc to find spatially-mismatched pairs (probably no good counterpart on the other side) or by area-diff to find the polygons that changed the most.

In [ ]:
# Show the largest-area disagreements first — those are usually the
# scientifically most consequential.
interesting = (match[match['disagree']]
               .sort_values('new_area_sqkm', ascending=False)
               .head(20))
cols = ['new_polyid','ref_polyid','match_km',
        'ref_area_sqkm','new_area_sqkm',
        'ref_alg_agg','new_alg_agg',
        'ref_v_lct','new_v_lct',
        'ref_v_tree','new_v_tree']
interesting[[c for c in cols if c in interesting.columns]]

### Zoom to a single fire

Pick one `new_polyid` from the table above and render a focused map showing just that polygon and its reference counterpart.

In [ ]:
# === EDIT: pick a polyid from the table above ===
#FOCUS_NEW_POLYID = int(interesting['new_polyid'].iloc[0]) if len(interesting) else None
FOCUS_NEW_POLYID = 4160

if FOCUS_NEW_POLYID is None:
    print('no disagreements in this subset to zoom into')
else:
    row = match[match['new_polyid'] == FOCUS_NEW_POLYID].iloc[0]
    focus_new = new_sub[new_sub['polyid'] == FOCUS_NEW_POLYID]
    focus_ref = ref_sub[ref_sub['polyid'] == row['ref_polyid']]
    print(f'NEW polyid={FOCUS_NEW_POLYID}   ↔   REF polyid={row["ref_polyid"]}'
          f'   distance={row["match_km"]:.3f} km')
    print()
    side_by_side = pd.DataFrame({
        'REF': {c: row[f'ref_{c}'] for c in ['area_sqkm','alg_agg','v_lct','v_tree','v_frp']
                if f'ref_{c}' in row.index},
        'NEW': {c: row[f'new_{c}'] for c in ['area_sqkm','alg_agg','v_lct','v_tree','v_frp']
                if f'new_{c}' in row.index},
    })
    print(side_by_side)

    # Focused zoom map
    pair_bounds = pd.concat([focus_ref, focus_new]).total_bounds
    pad = max(0.01, (pair_bounds[2]-pair_bounds[0]) * 0.5)
    pair_bounds = (pair_bounds[0]-pad, pair_bounds[1]-pad,
                   pair_bounds[2]+pad, pair_bounds[3]+pad)
    fcent = [(pair_bounds[1]+pair_bounds[3])/2, (pair_bounds[0]+pair_bounds[2])/2]
    m3 = folium.Map(location=fcent, zoom_start=12, tiles='OpenStreetMap')
    gdf_to_layer(focus_ref, 'REF (FINN2)', 'blue',  POPUP_FIELDS).add_to(m3)
    gdf_to_layer(focus_new, 'NEW (finn_py)','red',   POPUP_FIELDS).add_to(m3)
    # Add AF detections that fall inside the zoomed bbox.
    if af_sub is not None and len(af_sub):
        w, s, e, n = pair_bounds
        af_focus = af_sub.cx[w:e, s:n]
        af_layer = af_to_layer(af_focus, name=f'AF detections ({len(af_focus):,})',
                                radius=5)
        if af_layer is not None:
            af_layer.add_to(m3)
    LayerControl(collapsed=False).add_to(m3)
    m3.fit_bounds([[pair_bounds[1], pair_bounds[0]], [pair_bounds[3], pair_bounds[2]]])
    m3

## 8. Tips for using these maps

- **Zoom / pan** with mouse wheel and drag.
- **Layer control** (top-right) toggles each dataset on/off — flip between REF and NEW to compare side-by-side.
- **Hover** any polygon for a quick tooltip with the key fields.
- **Click** any polygon for a popup with the full attribute set.
- **Saved HTML files** in `HTML_OUT_DIR` are self-contained — share them with collaborators who don't have Jupyter.  Each map keeps interactivity intact when opened in any browser.
- **Subset bigger** by raising `MAX_POLYGONS` or relaxing the `VIEW_DATE` to a range — but expect noticeable lag past ~10k features per layer.
- **Different basemap** with `tiles=` on `folium.Map(...)` — useful options: `'CartoDB positron'` (light), `'Stamen Terrain'` (relief), or any XYZ tile server URL.